In [ ]:
%matplotlib widget
import numpy 
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets
import os
import cryspy
from ipyfilechooser import FileChooser

In [ ]:
def calc_gamma_nu_wavelength_for_hkl(h, k, l, UB, R):
    hkl = numpy.vstack([h, k, l]).T
    Q = (R @ (UB @ hkl.T)).T
    Qnorm = numpy.linalg.norm(Q, axis=1)
    cos_alpha = -Q[:,2]/Qnorm
    wavelength = 4*numpy.pi * cos_alpha / Qnorm
    ki = numpy.zeros(Q.shape,dtype=float)
    ki[:,2] = 2*numpy.pi/wavelength
    kf = ki + Q
    kf_x, kf_y, kf_z = kf[:,0], kf[:,1], kf[:,2]
    
    r = numpy.linalg.norm(kf, axis=1)

    gamma = numpy.rad2deg(numpy.arctan2(kf_x, kf_z))      # horizontal angle
    nu    = numpy.rad2deg(numpy.arcsin(kf_y / r))  
    return gamma, nu, wavelength


In [ ]:
def generate_peak_data(UB, R, hmax, kmax, lmax, lambda_min, lambda_max):
    """
    Generate synthetic diffraction peak data based on:
    - UB matrix (3x3)
    - crystal rotation matrix R (3x3)
    - HKL range: -hmax..hmax etc.
    - wavelength range (lambda_min, lambda_max)

    Returns array with columns:
    [h, k, l, gamma_deg, nu_deg, wavelength]
    """

    # --- 1. Generate HKL grid ---
    h = numpy.arange(-hmax, hmax+1)
    k = numpy.arange(-kmax, kmax+1)
    l = numpy.arange(-lmax, lmax+1)
    H, K, L = numpy.meshgrid(h, k, l, indexing='ij')
    hkl = numpy.vstack([H.ravel(), K.ravel(), L.ravel()]).T

    # Remove (0,0,0)
    hkl = hkl[numpy.any(hkl != 0, axis=1)]

    # --- 2. Compute Q vectors in lab frame ---
    # Apply UB and then rotation R
    Q = (R @ (UB @ hkl.T)).T

    # Magnitude of Q
    Qnorm = numpy.linalg.norm(Q, axis=1)


    # --- 3. Compute wavelength from Bragg condition ---
    cos_alpha = -Q[:,2]/Qnorm

    wavelength = 4*numpy.pi * cos_alpha / Qnorm

    # wavelength = 4*numpy.pi / Qnorm

    # --- 4. Apply wavelength limits ---
    mask = (wavelength >= lambda_min) & (wavelength <= lambda_max)
    hkl = hkl[mask]
    Q = Q[mask]
    wavelength = wavelength[mask]

    # --- 5. Convert Q direction to detector angles (γ, ν) ---
    ki = numpy.zeros(Q.shape,dtype=float)
    ki[:,2] = 2*numpy.pi/wavelength
    kf = ki + Q
    kf_x, kf_y, kf_z = kf[:,0], kf[:,1], kf[:,2]
    
    r = numpy.linalg.norm(kf, axis=1)

    gamma = numpy.rad2deg(numpy.arctan2(kf_x, kf_z))      # horizontal angle
    nu    = numpy.rad2deg(numpy.arcsin(kf_y / r))       # vertical angle

    # --- 6. Build final array ---
    result = numpy.column_stack([hkl[:,0], hkl[:,1], hkl[:,2], gamma, nu, wavelength])
    return result




In [ ]:
def calc_orientation_matrix(euler_alpha, euler_beta, euler_gamma, ):
    ca, cb, cg = numpy.cos(euler_alpha), numpy.cos(euler_beta), numpy.cos(euler_gamma)
    sa, sb, sg = numpy.sin(euler_alpha), numpy.sin(euler_beta), numpy.sin(euler_gamma)
    m_m = numpy.array([
        [ca*cb, ca*sb*sg-sa*cg, ca*sb*cg+sa*sg],
        [sa*cb, sa*sb*sg+ca*cg, sa*sb*cg-ca*sg],
        [-sb, cb*sg, cb*cg],
    ], dtype=float)
    return m_m

In [ ]:

def calc_sample_rotation(sample_omega, sample_chi, sample_phi):
    zero_o = numpy.sin(numpy.zeros_like(sample_omega))
    one_o = numpy.cos(numpy.zeros_like(sample_omega))
    m_omega = numpy.array([
        [numpy.cos(sample_omega), zero_o, numpy.sin(sample_omega)],
        [zero_o, one_o, zero_o],
        [-numpy.sin(sample_omega), zero_o, numpy.cos(sample_omega)],
    ],dtype=float)
    zero_c = numpy.sin(numpy.zeros_like(sample_chi))
    one_c = numpy.cos(numpy.zeros_like(sample_chi))
    m_chi = numpy.array([
        [numpy.cos(sample_chi), -numpy.sin(sample_chi), zero_c],
        [numpy.sin(sample_chi), numpy.cos(sample_chi), zero_c],
        [zero_c, zero_c, one_c],
    ],dtype=float)

    zero_p = numpy.sin(numpy.zeros_like(sample_phi))
    one_p = numpy.cos(numpy.zeros_like(sample_phi))
    m_phi = numpy.array([
        [numpy.cos(sample_phi), zero_p, numpy.sin(sample_phi)],
        [zero_p, one_p, zero_p],
        [-numpy.sin(sample_phi), zero_p, numpy.cos(sample_phi)],
    ], dtype=float)
    sample_rotation = m_omega @ m_chi @ m_phi
    return sample_rotation


In [ ]:

def _calc_cell_phi(cell_alpha, cell_beta, cell_gamma):
    ca, cb, cg = numpy.cos(cell_alpha), numpy.cos(cell_beta), numpy.cos(cell_gamma)
    cell_phi = numpy.sqrt(1. - ca*ca - cb*cb - cg*cg + 2 * ca * cb * cg)
    return cell_phi

def calc_cell_volume(cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma):
    cell_phi = _calc_cell_phi(cell_alpha, cell_beta, cell_gamma)
    cell_volume = cell_a * cell_b * cell_c * cell_phi
    return cell_volume

def calc_b_matrix(cell_a, cell_b, cell_c, cell_alpha, cell_beta, cell_gamma):
    cell_phi = _calc_cell_phi(cell_alpha, cell_beta, cell_gamma)
    a, b, c = cell_a, cell_b, cell_c
    b_11 = numpy.sin(cell_alpha)/(a*cell_phi)
    b_12 = (numpy.cos(cell_alpha)*numpy.cos(cell_beta)-numpy.cos(cell_gamma))/(b*cell_phi*numpy.sin(cell_alpha))
    b_13 = (numpy.cos(cell_alpha)*numpy.cos(cell_gamma)-numpy.cos(cell_beta))/(c*cell_phi*numpy.sin(cell_alpha))
    b_22 = 1/(b*numpy.sin(cell_alpha))
    b_23 = -numpy.cos(cell_alpha)/(c*numpy.sin(cell_alpha))
    b_33 = 1/c
    zero = 0.
    b_matrix = numpy.array([
            [b_11, b_12, b_13],
            [zero, b_22, b_23],
            [zero, zero, b_33],
        ],dtype=float)
    return b_matrix

In [ ]:
def hkl_to_rgb(h,k,l):
    hkl = numpy.sqrt(numpy.square(h) + numpy.square(k) + numpy.square(l))
    hh_r = numpy.abs(h) / hkl
    hh_g = numpy.abs(k) / hkl
    hh_b = numpy.abs(l) / hkl
    return numpy.stack([hh_r, hh_g, hh_b], axis=1)

In [ ]:

# ---------------------------------------------------------
# INPUT: your array with columns (h, k, l, gamma, nu, wavelength)
# ---------------------------------------------------------
# Example placeholder:
# data = numpy.load("your_array.npy")
# Columns: 0=h, 1=k, 2=l, 3=gamma, 4=nu, 5=wavelength

# For demonstration, create dummy data:
# data = numpy.random.rand(100,6)


# ---------------------------------------------------------
# Plotting function
# ---------------------------------------------------------
def plot_hkl(gamma_min_max, nu_min_max, lambda_min_max, h_min_max, k_min_max, l_min_max):
    if not 'data' in globals():
        return
    plt.figure(figsize=(8,6))

    # Mask based on user limits
    mask1 = numpy.logical_and(data[:,3] >= gamma_min_max[0], data[:,3] <= gamma_min_max[1])
    d = data[mask1]
    if d.shape[1]>=7:
        mask = d[:,6] >= 0.01*d[:,6].max()
        d = d[mask]
    mask2 = numpy.logical_and(d[:,4] >= nu_min_max[0], d[:,4] <= nu_min_max[1])
    d = d[mask2]
    mask3 = numpy.logical_and(d[:,5] >= lambda_min_max[0], d[:,5] <= lambda_min_max[1])
    d = d[mask3]
    mask4 = numpy.logical_and(d[:,0] >= -h_min_max, d[:,0] <= h_min_max)
    d = d[mask4]
    mask5 = numpy.logical_and(d[:,1] >= -k_min_max, d[:,1] <= k_min_max)
    d = d[mask5]
    mask6 = numpy.logical_and(d[:,2] >= -l_min_max, d[:,2] <= l_min_max)
    d = d[mask6]

    size = 4
    if d.shape[1]>=7:
        size = 20*size*d[:,6]/d[:,6].max()
    
    np_rgb = hkl_to_rgb(d[:,0], d[:,1], d[:,2])

    # Scatter plot
    ax = plt.scatter(d[:,3], d[:,4], s=size, color=np_rgb)
    d_h = d[numpy.argsort(np_rgb[:,0], descending=True)[:4],:]
    d_k = d[numpy.argsort(np_rgb[:,1], descending=True)[:4],:]
    d_l = d[numpy.argsort(np_rgb[:,2], descending=True)[:4],:]
    d_hkl = d[numpy.argsort(np_rgb.sum(axis=1))[:4],:]

    # Label each point with (hkl)
    if d.shape[1]>=7:
        pass
        for h, k, l, g, n, lam in d[numpy.argsort(size,descending=True)[:10],:6]:
            plt.text(g, n, f"({int(h)} {int(k)} {int(l)})", fontsize=8)
    else:
        for dd in [d_h, d_k, d_l, d_hkl]:
            for h, k, l, g, n, lam in dd[:,:6]:
                plt.text(g, n, f"({int(h)} {int(k)} {int(l)})", fontsize=8)

    plt.xlabel("gamma (deg)")
    plt.ylabel("nu (deg)")
    plt.xlim((gamma_min_max[0],gamma_min_max[1]))
    plt.ylim((nu_min_max[0], nu_min_max[1]))
    
    plt.title("Peak Positions")
    plt.grid(True)
    plt.show()


# ---------------------------------------------------------
# Widgets
# ---------------------------------------------------------
gamma_min_max = ipywidgets.FloatRangeSlider(description="γ min/max", value=[0, 90], min=-180, max=180, step=0.1, orientation='horizontal', readout=True, readout_format='.1f',)
nu_min_max = ipywidgets.FloatRangeSlider(description="ν min/max", value=[-15, 45], min=-90, max=90, step=0.1, orientation='horizontal', readout=True, readout_format='.1f',)
lambda_min_max = ipywidgets.FloatRangeSlider(description="λ min/max", value=[0.5, 6], min=0.1, max=10, step=0.01, orientation='horizontal', readout=True, readout_format='.2f',)


h_min_max = ipywidgets.IntSlider(description="H max", value=100, min=0, max=100, step=1, orientation='horizontal', readout=True,)
k_min_max = ipywidgets.IntSlider(description="K max", value=100, min=0, max=100, step=1, orientation='horizontal', readout=True,)
l_min_max = ipywidgets.IntSlider(description="L max", value=100, min=0, max=100, step=1, orientation='horizontal', readout=True,)



In [ ]:


# --- your functions here ---
# calc_b_matrix(...)
# calc_orientation_matrix(...)
# calc_sample_rotation(...)
# generate_peak_data(...)

# ---------------------------
# Widgets
# ---------------------------
dir_path_data = "/Users/iuriikibalin/Desktop"

if not os.path.isdir(dir_path_data):
    dir_path_data = "/"

fc = FileChooser(
    # '/Users/iuriikibalin/Downloads/C60_n3',
    dir_path_data,
    title="Select CIF File (Optional)",
    select_default=False,
)



ea_alpha_w = ipywidgets.FloatText(description="α (deg)", value=0.0)
ea_beta_w  = ipywidgets.FloatText(description="β (deg)", value=0.0)
ea_gamma_w = ipywidgets.FloatText(description="γ (deg)", value=0.0)

omega_w = ipywidgets.FloatText(description="ω (deg)", value=0.0)
chi_w   = ipywidgets.FloatText(description="χ (deg)", value=0.0)
phi_w   = ipywidgets.FloatText(description="φ (deg)", value=0.0)

lambda_min_w = ipywidgets.FloatText(description="λ_min", value=0.5)
lambda_max_w = ipywidgets.FloatText(description="λ_max", value=6.0)

cell_a_w = ipywidgets.FloatText(description="a", value=14.0)
cell_b_w = ipywidgets.FloatText(description="b", value=14.0)
cell_c_w = ipywidgets.FloatText(description="c", value=14.0)

cell_alpha_w = ipywidgets.FloatText(description="α (deg)", value=90.0)
cell_beta_w  = ipywidgets.FloatText(description="β (deg)", value=90.0)
cell_gamma_w = ipywidgets.FloatText(description="γ (deg)", value=90.0)


# out = ipywidgets.Output()

out = ipywidgets.interactive_output(
    plot_hkl,
    {
        "gamma_min_max": gamma_min_max,
        "nu_min_max": nu_min_max,
        "lambda_min_max": lambda_min_max,
        "h_min_max": h_min_max,
        "k_min_max": k_min_max,
        "l_min_max": l_min_max,
    }
)

button = ipywidgets.Button(description="Generate peaks", button_style="success")

hkl_w = ipywidgets.Text(
    value='',
    placeholder='2 0 0',
    description='HKL:',
    disabled=False
)
button_hkl = ipywidgets.Button(description="Calc angles for HKL", button_style="success")

# ---------------------------
# Callback
# ---------------------------

def compute_angles(_):
    with out:
        out.clear_output()
        cell_a = cell_a_w.value
        cell_b = cell_b_w.value
        cell_c = cell_c_w.value

        cell_alpha = numpy.deg2rad(cell_alpha_w.value)
        cell_beta  = numpy.deg2rad(cell_beta_w.value)
        cell_gamma = numpy.deg2rad(cell_gamma_w.value)

        ea_alpha = numpy.deg2rad(ea_alpha_w.value)
        ea_beta  = numpy.deg2rad(ea_beta_w.value)
        ea_gamma = numpy.deg2rad(ea_gamma_w.value)

        sample_omega = numpy.deg2rad(omega_w.value)
        sample_chi   = numpy.deg2rad(chi_w.value)
        sample_phi   = numpy.deg2rad(phi_w.value)

        # --- compute matrices ---
        m_b = calc_b_matrix(cell_a, cell_b, cell_c,
                            cell_alpha, cell_beta, cell_gamma)

        m_u = calc_orientation_matrix(ea_alpha, ea_beta, ea_gamma)
        m_ub = m_u @ m_b

        m_r = calc_sample_rotation(sample_omega, sample_chi, sample_phi)
        s_hkl = hkl_w.value
        try:
            (h,k,l) = (float(hh) for hh in s_hkl.strip().split()[:3])
        except:
            s_hkl = hkl_w.placeholder
            (h,k,l) = (float(hh) for hh in s_hkl.strip().split()[:3])

        gamma, nu, wavelength = calc_gamma_nu_wavelength_for_hkl(h,k,l, m_ub, m_r)
        print(f'# Peak ({h:.1f} {k:.1f} {l:.1f})') 
        print(f"\nUB matrix: \n{m_ub[0,0]:9.5f}{m_ub[0,1]:9.5f}{m_ub[0,1]:9.5f}\n{m_ub[1,0]:9.5f}{m_ub[1,1]:9.5f}{m_ub[1,2]:9.5f}\n{m_ub[2,0]:9.5f}{m_ub[2,1]:9.5f}{m_ub[2,2]:9.5f}")
        print(f"\nRotation matrix: \n{m_r[0,0]:9.5f}{m_r[0,1]:9.5f}{m_r[0,1]:9.5f}\n{m_r[1,0]:9.5f}{m_r[1,1]:9.5f}{m_r[1,2]:9.5f}\n{m_r[2,0]:9.5f}{m_r[2,1]:9.5f}{m_r[2,2]:9.5f}")
        print(f"\nGamma is {gamma.squeeze():.2f} deg.\nNu is {nu.squeeze():.2f} deg.\nWavelength is {wavelength.squeeze():.5f} Ang.")

def on_file_selected(_):
    file_path = fc.selected

    with out:
        out.clear_output()
        print(file_path is None or not os.path.isfile(file_path))
        if file_path is None or not os.path.isfile(file_path):
            # --- read inputs ---
            pass
        else:
            rcif_obj = cryspy.file_to_globaln(file_path)          
            crystal = [hh for hh in rcif_obj.items if isinstance(hh, cryspy.Crystal)][0]
            ucp = crystal.get_dictionary()['unit_cell_parameters']
            cell_a_w.value = ucp[0]
            cell_b_w.value = ucp[1]
            cell_c_w.value = ucp[2]
            cell_alpha_w.value = numpy.rad2deg(ucp[3])
            cell_beta_w.value = numpy.rad2deg(ucp[4])
            cell_gamma_w.value = numpy.rad2deg(ucp[5])

    return


def compute(_):
    with out:
        global data
        out.clear_output()

        file_path = fc.selected
        flag_crystal = False
        if file_path is None or not os.path.isfile(file_path):
            # --- read inputs ---
            pass
        else:
            try:
                rcif_obj = cryspy.file_to_globaln(file_path)            
                crystal = [hh for hh in rcif_obj.items if isinstance(hh, cryspy.Crystal)][0]
                flag_crystal = True
            except:
                flag_crystal = False

        cell_a = cell_a_w.value
        cell_b = cell_b_w.value
        cell_c = cell_c_w.value
        cell_alpha = numpy.deg2rad(cell_alpha_w.value)
        cell_beta = numpy.deg2rad(cell_beta_w.value)
        cell_gamma = numpy.deg2rad(cell_gamma_w.value)



        ea_alpha = numpy.deg2rad(ea_alpha_w.value)
        ea_beta  = numpy.deg2rad(ea_beta_w.value)
        ea_gamma = numpy.deg2rad(ea_gamma_w.value)

        sample_omega = numpy.deg2rad(omega_w.value)
        sample_chi   = numpy.deg2rad(chi_w.value)
        sample_phi   = numpy.deg2rad(phi_w.value)

        lambda_min = lambda_min_w.value
        lambda_max = lambda_max_w.value

        # --- compute limits ---
        hmax = 2 * cell_a / lambda_min
        kmax = 2 * cell_b / lambda_min
        lmax = 2 * cell_c / lambda_min

        # --- compute matrices ---
        m_b = calc_b_matrix(cell_a, cell_b, cell_c,
                            cell_alpha, cell_beta, cell_gamma)

        m_u = calc_orientation_matrix(ea_alpha, ea_beta, ea_gamma)
        m_ub = m_u @ m_b

        m_r = calc_sample_rotation(sample_omega, sample_chi, sample_phi)

        # --- generate peak data ---
        data = generate_peak_data(
            UB=m_ub,
            R=m_r,
            hmax=hmax, kmax=kmax, lmax=lmax,
            lambda_min=lambda_min,
            lambda_max=lambda_max
        )
        if flag_crystal:
            fn = crystal.calc_f_nucl(numpy.stack([data[:,0], data[:,1], data[:,2]], axis=0))
            fn_sq = numpy.square(numpy.abs(fn))
            data = numpy.column_stack((data, fn_sq))
            print(fn_sq)
        # --- output ---
        # print("UB matrix:\n", m_ub)
        # print("\nRotation matrix R:\n", m_r)
        # print("\nNumber of peaks:", len(data))
        # print(f"\nMin, max HKL: {min(data[:,0]):.0f}, {max(data[:,0]):.0f}; {min(data[:,1]):.0f}, {max(data[:,1]):.0f}; {min(data[:,2]):.0f}, {max(data[:,2]):.0f}")

        cos_tth = numpy.cos(numpy.radians(data[:,3]))*numpy.cos(numpy.radians(data[:,4]))
        
        # data_sort = data[numpy.argsort(cos_tth, descending=True),:]
        # print("\nFirst few peaks:\n", numpy.round(data_sort[:10],2))
        plot_hkl(gamma_min_max=gamma_min_max.value,nu_min_max=nu_min_max.value,lambda_min_max=lambda_min_max.value,
                 h_min_max=h_min_max.value, k_min_max=k_min_max.value, l_min_max=l_min_max.value)


button.on_click(compute)
button_hkl.on_click(compute_angles)

fc.register_callback(on_file_selected)
# ---------------------------
# Layout for Voila
# ---------------------------


ui = ipywidgets.VBox([
    fc, 
    ipywidgets.HTML("<h3>Diffraction Peaks on detector</h3>"),
    ipywidgets.HTML("<h4>Unit cell parameters</h4>"),
    ipywidgets.HBox([cell_a_w, cell_b_w, cell_c_w]),
    ipywidgets.HBox([cell_alpha_w, cell_beta_w, cell_gamma_w]),
    ipywidgets.HTML("<h4>Sample Orientation (Euleur angles)</h4>"),
    ipywidgets.HBox([ea_alpha_w, ea_beta_w, ea_gamma_w]),
    ipywidgets.HTML("<h4>Sample Rotation</h4>"),
    ipywidgets.HBox([omega_w, chi_w, phi_w]),
    ipywidgets.HTML("<h4>Wavelength Range</h4>"),
    ipywidgets.HBox([lambda_min_w, lambda_max_w]),
    ipywidgets.HTML("<h4> </h4>"),
    button,
    ipywidgets.HBox([button_hkl, hkl_w]),
    ipywidgets.HTML("<h4>Graph Filters</h4>"),
    ipywidgets.HBox([gamma_min_max,nu_min_max,lambda_min_max,]),
    ipywidgets.HBox([h_min_max,k_min_max,l_min_max,]),
    out,
])

display(ui)